In [ ]:
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
import random
from collections import defaultdict
import csv
import json

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==================== CONFIGURATION ====================

# Base directory
base_dir = '/content/drive/MyDrive/Final_Year/Final_Year_Project/Google_Colab/lk_dataset_manual_processing'

# Source dataset path in Google Drive
SOURCE_PATH = f"{base_dir}/lk_dataset_manual_correction/lk_acts"  # Update this path

# Destination path for the split dataset
DEST_PATH = f"{base_dir}/sinhala_ocr_dataset_1010_stratified"  # Update this path

In [ ]:
def get_split_ratios():
    """
    Get split ratios from user input.
    Returns: dict with 'train', 'eval', 'test' percentages
    """
    print("DATASET SPLIT CONFIGURATION")
    print("-" * 40)

    split_type = input("\nChoose split type:\n1. Train/Eval/Test (3-way split)\n2. Train/Test (2-way split)\nEnter choice (1 or 2): ").strip()

    if split_type == '1':
        while True:
            try:
                train = float(input("Enter training percentage (e.g., 70): "))
                eval_pct = float(input("Enter evaluation percentage (e.g., 10): "))
                test = float(input("Enter testing percentage (e.g., 20): "))

                if abs(train + eval_pct + test - 100) < 0.01:
                    return {
                        'train': train / 100,
                        'eval': eval_pct / 100,
                        'test': test / 100,
                        'use_eval': True
                    }
                else:
                    print("Error: Percentages must sum to 100. Please try again.")
            except ValueError:
                print("Error: Please enter valid numbers.")

    elif split_type == '2':
        while True:
            try:
                train = float(input("Enter training percentage (e.g., 80): "))
                test = 100 - train
                print(f"Testing percentage will be: {test}%")

                return {
                    'train': train / 100,
                    'eval': 0,
                    'test': test / 100,
                    'use_eval': False
                }
            except ValueError:
                print("Error: Please enter a valid number.")
    else:
        print("Invalid choice. Using default 70/10/20 split.")
        return {'train': 0.7, 'eval': 0.1, 'test': 0.2, 'use_eval': True}

In [ ]:
def collect_page_pairs(source_path):
    """
    Collect all image-text pairs from the dataset.
    Returns: list of tuples (image_path, text_path, year, doc_id, page_num)
    """
    page_pairs = []
    source_path = Path(source_path)

    for decade_folder in source_path.iterdir():
        if not decade_folder.is_dir():
            continue

        for year_folder in decade_folder.iterdir():
            if not year_folder.is_dir():
                continue

            year_name = year_folder.name
            page_images_dir = year_folder / 'page_images'
            pages_text_dir = year_folder / 'pages_text'

            if not page_images_dir.exists() or not pages_text_dir.exists():
                continue

            for doc_folder in page_images_dir.iterdir():
                if not doc_folder.is_dir() or doc_folder.name == '.':
                    continue

                doc_id = doc_folder.name
                text_doc_folder = pages_text_dir / doc_id

                if not text_doc_folder.exists():
                    continue

                for img_file in doc_folder.iterdir():
                    if img_file.suffix.lower() == '.png':
                        page_num = img_file.stem
                        txt_file = text_doc_folder / f"{page_num}.txt"

                        if txt_file.exists():
                            page_pairs.append({
                                'image': str(img_file),
                                'text': str(txt_file),
                                'year': year_name,
                                'doc_id': doc_id,
                                'page_num': page_num
                            })

    return page_pairs

In [ ]:
def split_dataset(page_pairs, ratios):
    """
    Split dataset into train/eval/test sets.
    """
    random.seed(42)
    random.shuffle(page_pairs)

    total = len(page_pairs)
    train_size = int(total * ratios['train'])

    if ratios['use_eval']:
        eval_size = int(total * ratios['eval'])
        test_size = total - train_size - eval_size

        train_data = page_pairs[:train_size]
        eval_data = page_pairs[train_size:train_size + eval_size]
        test_data = page_pairs[train_size + eval_size:]

        return {
            'train': train_data,
            'eval': eval_data,
            'test': test_data
        }
    else:
        test_size = total - train_size

        train_data = page_pairs[:train_size]
        test_data = page_pairs[train_size:]

        return {
            'train': train_data,
            'test': test_data
        }

In [ ]:
def create_split_dataset(splits, dest_path):
    """
    Create the new dataset structure with split data.
    """
    dest_path = Path(dest_path)

    if dest_path.exists():
        response = input(f"\nDestination '{dest_path}' already exists. Overwrite? (yes/no): ")
        if response.lower() != 'yes':
            print("Aborted.")
            return
        shutil.rmtree(dest_path)

    dest_path.mkdir(parents=True, exist_ok=True)

    for split_name, pages in splits.items():
        print(f"Creating {split_name} set with {len(pages)} pages...")

        split_dir = dest_path / split_name
        images_dir = split_dir / 'images'
        labels_dir = split_dir / 'labels'

        images_dir.mkdir(parents=True, exist_ok=True)
        labels_dir.mkdir(parents=True, exist_ok=True)

        for idx, page in enumerate(pages, 1):
            filename = f"{page['year']}_{page['doc_id']}_{page['page_num']}"

            img_dest = images_dir / f"{filename}.png"
            shutil.copy2(page['image'], img_dest)

            txt_dest = labels_dir / f"{filename}.txt"
            shutil.copy2(page['text'], txt_dest)

            if idx % 100 == 0:
                print(f"  Processed {idx}/{len(pages)} pages...")

        print(f"Completed {split_name} set: {len(pages)} pages")

In [ ]:
def create_metadata(splits, dest_path):
    """
    Create comprehensive metadata files for each split including year tracking.
    Creates TXT, CSV, JSON, and year breakdown formats.
    """
    dest_path = Path(dest_path)

    for split_name, pages in splits.items():
        split_dir = dest_path / split_name

        # Create detailed TXT metadata
        metadata_txt = split_dir / 'metadata.txt'
        with open(metadata_txt, 'w', encoding='utf-8') as f:
            f.write(f"Split: {split_name}\n")
            f.write(f"Total pages: {len(pages)}\n")
            f.write(f"{'-' * 70}\n\n")

            for page in pages:
                filename = f"{page['year']}_{page['doc_id']}_{page['page_num']}"
                f.write(f"Filename: {filename}\n")
                f.write(f"  Image: images/{filename}.png\n")
                f.write(f"  Label: labels/{filename}.txt\n")
                f.write(f"  Year: {page['year']}\n")
                f.write(f"  Document: {page['doc_id']}\n")
                f.write(f"  Page: {page['page_num']}\n")
                f.write("-" * 70 + "\n")

        # Create CSV metadata
        metadata_csv = split_dir / 'metadata.csv'
        with open(metadata_csv, 'w', encoding='utf-8', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['filename', 'image_path', 'label_path', 'year', 'doc_id', 'page_num'])
            writer.writeheader()

            for page in pages:
                filename = f"{page['year']}_{page['doc_id']}_{page['page_num']}"
                writer.writerow({
                    'filename': filename,
                    'image_path': f"images/{filename}.png",
                    'label_path': f"labels/{filename}.txt",
                    'year': page['year'],
                    'doc_id': page['doc_id'],
                    'page_num': page['page_num']
                })

        # Create JSON metadata
        metadata_json = split_dir / 'metadata.json'
        json_data = {
            'split': split_name,
            'total_pages': len(pages),
            'samples': []
        }

        for page in pages:
            filename = f"{page['year']}_{page['doc_id']}_{page['page_num']}"
            json_data['samples'].append({
                'filename': filename,
                'image_path': f"images/{filename}.png",
                'label_path': f"labels/{filename}.txt",
                'year': page['year'],
                'doc_id': page['doc_id'],
                'page_num': page['page_num']
            })

        with open(metadata_json, 'w', encoding='utf-8') as f:
            json.dump(json_data, f, indent=2, ensure_ascii=False)

        # Create year-wise breakdown
        year_breakdown = split_dir / 'year_breakdown.txt'
        year_stats = defaultdict(int)
        for page in pages:
            year_stats[page['year']] += 1

        with open(year_breakdown, 'w', encoding='utf-8') as f:
            f.write(f"Year-wise Distribution for {split_name.upper()} Set\n")
            f.write(f"{'-' * 50}\n\n")
            f.write(f"{'Year':<10} {'Count':<10} {'Percentage':<15}\n")
            f.write(f"{'-' * 50}\n")

            for year in sorted(year_stats.keys()):
                count = year_stats[year]
                percentage = (count / len(pages)) * 100
                f.write(f"{year:<10} {count:<10} {percentage:.2f}%\n")

            f.write(f"{'-' * 50}\n")
            f.write(f"{'TOTAL':<10} {len(pages):<10} 100.00%\n")

        print(f"Created metadata files for {split_name} set")

In [ ]:
def print_summary(page_pairs, splits):
    """
    Print summary statistics of the dataset split.
    """
    print("\n" + "-" * 70)
    print("DATASET SPLIT SUMMARY")
    print("-" * 70)
    print(f"\nTotal pages collected: {len(page_pairs)}")
    print(f"\nSplit distribution:")

    for split_name, pages in splits.items():
        percentage = (len(pages) / len(page_pairs)) * 100
        print(f"  {split_name.capitalize():10s}: {len(pages):5d} pages ({percentage:.1f}%)")

    print(f"\n" + "-" * 70)
    print("YEAR-WISE DISTRIBUTION BY SPLIT")
    print("-" * 70)

    for split_name, pages in splits.items():
        year_stats = defaultdict(int)
        for page in pages:
            year_stats[page['year']] += 1

        print(f"\n{split_name.upper()} Set:")
        for year in sorted(year_stats.keys()):
            count = year_stats[year]
            percentage = (count / len(pages)) * 100
            print(f"  {year}: {count:4d} pages ({percentage:5.2f}%)")

    print("\n" + "-" * 70)

In [ ]:
def main():
    """
    Main execution function.
    """
    print("\nSRI LANKAN ACTS OCR DATASET PREPARATION")
    print("-" * 70)

    ratios = get_split_ratios()

    print(f"\nScanning dataset from: {SOURCE_PATH}")
    page_pairs = collect_page_pairs(SOURCE_PATH)
    print(f"Found {len(page_pairs)} page pairs")

    if len(page_pairs) == 0:
        print("No data found. Please check the source path.")
        return

    print(f"\nSplitting dataset...")
    splits = split_dataset(page_pairs, ratios)

    print(f"\nCreating split dataset at: {DEST_PATH}")
    create_split_dataset(splits, DEST_PATH)

    print(f"\nCreating metadata files...")
    create_metadata(splits, DEST_PATH)

    print_summary(page_pairs, splits)

    print(f"\nDataset preparation complete!")
    print(f"Output location: {DEST_PATH}")
    print("\nFolder structure:")
    print(f"  {DEST_PATH}/")
    for split_name in splits.keys():
        print(f"    {split_name}/")
        print(f"      ├── images/")
        print(f"      ├── labels/")
        print(f"      ├── metadata.txt")
        print(f"      ├── metadata.csv")
        print(f"      ├── metadata.json")
        print(f"      └── year_breakdown.txt")

if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path

def verify_dataset_count(source_path):
    """
    Count total images and text files in the dataset.
    """
    source_path = Path(source_path)

    total_images = 0
    total_texts = 0
    paired_count = 0

    print("Verifying dataset file counts...\n")

    for decade_folder in source_path.iterdir():
        if not decade_folder.is_dir():
            continue

        for year_folder in decade_folder.iterdir():
            if not year_folder.is_dir():
                continue

            page_images_dir = year_folder / 'page_images'
            pages_text_dir = year_folder / 'pages_text'

            if page_images_dir.exists():
                for doc_folder in page_images_dir.iterdir():
                    if doc_folder.is_dir() and doc_folder.name != '.':
                        images = list(doc_folder.glob('*.png'))
                        total_images += len(images)

            if pages_text_dir.exists():
                for doc_folder in pages_text_dir.iterdir():
                    if doc_folder.is_dir() and doc_folder.name != '.':
                        texts = list(doc_folder.glob('*.txt'))
                        total_texts += len(texts)

            if page_images_dir.exists() and pages_text_dir.exists():
                for doc_folder in page_images_dir.iterdir():
                    if not doc_folder.is_dir() or doc_folder.name == '.':
                        continue

                    doc_id = doc_folder.name
                    text_doc_folder = pages_text_dir / doc_id

                    if text_doc_folder.exists():
                        for img_file in doc_folder.iterdir():
                            if img_file.suffix.lower() == '.png':
                                page_num = img_file.stem
                                txt_file = text_doc_folder / f"{page_num}.txt"
                                if txt_file.exists():
                                    paired_count += 1

    print("-" * 60)
    print("DATASET FILE COUNT VERIFICATION")
    print("-" * 60)
    print(f"\nTotal PNG images found:     {total_images:,}")
    print(f"Total TXT files found:      {total_texts:,}")
    print(f"Valid image-text pairs:     {paired_count:,}")

    if total_images != total_texts:
        print(f"\nWarning: Image count ({total_images}) doesn't match text count ({total_texts})")
    else:
        print(f"\nAll files are properly paired!")

    print("-" * 60)

    return {
        'images': total_images,
        'texts': total_texts,
        'pairs': paired_count
    }

counts = verify_dataset_count(SOURCE_PATH)

In [ ]:
from pathlib import Path

def verify_split_dataset(dest_path):
    """
    Verify file counts in the split dataset (train/eval/test).
    """
    dest_path = Path(dest_path)

    if not dest_path.exists():
        print(f"Destination path does not exist: {dest_path}")
        return None

    print("Verifying split dataset file counts...\n")

    split_stats = {}
    total_images = 0
    total_labels = 0
    total_pairs = 0

    for split_folder in dest_path.iterdir():
        if not split_folder.is_dir():
            continue

        split_name = split_folder.name
        images_dir = split_folder / 'images'
        labels_dir = split_folder / 'labels'

        if not images_dir.exists() or not labels_dir.exists():
            continue

        images = list(images_dir.glob('*.png'))
        labels = list(labels_dir.glob('*.txt'))

        image_count = len(images)
        label_count = len(labels)

        paired = 0
        for img_file in images:
            label_file = labels_dir / f"{img_file.stem}.txt"
            if label_file.exists():
                paired += 1

        split_stats[split_name] = {
            'images': image_count,
            'labels': label_count,
            'pairs': paired
        }

        total_images += image_count
        total_labels += label_count
        total_pairs += paired

    print("-" * 70)
    print("SPLIT DATASET VERIFICATION")
    print("-" * 70)
    print(f"\nLocation: {dest_path}\n")

    print(f"{'Split':<15} {'Images':<12} {'Labels':<12} {'Valid Pairs':<15} {'Status':<10}")
    print("-" * 70)

    for split_name in sorted(split_stats.keys()):
        stats = split_stats[split_name]
        status = "" if stats['images'] == stats['labels'] == stats['pairs'] else "warning"
        print(f"{split_name.capitalize():<15} {stats['images']:<12,} {stats['labels']:<12,} {stats['pairs']:<15,} {status:<10}")

    print("-" * 70)
    print(f"{'TOTAL':<15} {total_images:<12,} {total_labels:<12,} {total_pairs:<15,}")

    print("\n" + "-" * 70)
    print("SUMMARY")
    print("-" * 70)

    print(f"\nTotal Splits Found: {len(split_stats)}")
    print(f"Total Images: {total_images:,}")
    print(f"Total Labels: {total_labels:,}")
    print(f"Valid Pairs: {total_pairs:,}")

    all_valid = True
    issues = []

    for split_name, stats in split_stats.items():
        if stats['images'] != stats['labels']:
            all_valid = False
            issues.append(f"{split_name}: Image count ({stats['images']}) != Label count ({stats['labels']})")
        if stats['pairs'] != stats['images']:
            all_valid = False
            issues.append(f"{split_name}: Missing {stats['images'] - stats['pairs']} matching pairs")

    if all_valid and total_images == total_labels == total_pairs:
        print(f"\nAll files are properly paired and organized!")
    else:
        print(f"\nIssues detected:")
        for issue in issues:
            print(f"  {issue}")

    if total_images > 0:
        print(f"\nSplit Distribution:")
        for split_name in sorted(split_stats.keys()):
            percentage = (split_stats[split_name]['images'] / total_images) * 100
            print(f"  {split_name.capitalize():<10}: {percentage:>5.1f}%")

    print("\n" + "-" * 70)

    return split_stats


def compare_source_dest(source_path, dest_path):
    """
    Compare file counts between source and destination to ensure no data loss.
    """
    print("\n" + "-" * 70)
    print("SOURCE vs DESTINATION COMPARISON")
    print("-" * 70 + "\n")

    source_path = Path(source_path)
    source_count = 0

    for decade_folder in source_path.iterdir():
        if not decade_folder.is_dir():
            continue
        for year_folder in decade_folder.iterdir():
            if not year_folder.is_dir():
                continue
            page_images_dir = year_folder / 'page_images'
            pages_text_dir = year_folder / 'pages_text'

            if page_images_dir.exists() and pages_text_dir.exists():
                for doc_folder in page_images_dir.iterdir():
                    if not doc_folder.is_dir() or doc_folder.name == '.':
                        continue
                    doc_id = doc_folder.name
                    text_doc_folder = pages_text_dir / doc_id

                    if text_doc_folder.exists():
                        for img_file in doc_folder.iterdir():
                            if img_file.suffix.lower() == '.png':
                                page_num = img_file.stem
                                txt_file = text_doc_folder / f"{page_num}.txt"
                                if txt_file.exists():
                                    source_count += 1

    dest_path = Path(dest_path)
    dest_count = 0

    if dest_path.exists():
        for split_folder in dest_path.iterdir():
            if split_folder.is_dir():
                images_dir = split_folder / 'images'
                if images_dir.exists():
                    dest_count += len(list(images_dir.glob('*.png')))

    print(f"Source Dataset (Original):  {source_count:,} pairs")
    print(f"Destination Dataset (Split): {dest_count:,} pairs")

    if source_count == dest_count:
        print(f"\nPerfect match! All {source_count:,} samples were successfully split.")
    else:
        difference = abs(source_count - dest_count)
        print(f"\nMismatch detected!")
        if source_count > dest_count:
            print(f"Missing {difference:,} samples in destination")
        else:
            print(f"Extra {difference:,} samples in destination (unexpected)")

    print("\n" + "-" * 70 + "\n")

    return {
        'source': source_count,
        'destination': dest_count,
        'match': source_count == dest_count
    }


print("STEP 1: Verifying Split Dataset Structure\n")
split_stats = verify_split_dataset(DEST_PATH)

print("\nSTEP 2: Comparing Source vs Destination\n")
comparison = compare_source_dest(SOURCE_PATH, DEST_PATH)

print("-" * 70)
print("FINAL VERIFICATION REPORT")
print("-" * 70)

if split_stats and comparison:
    if comparison['match']:
        print("\nSUCCESS: Dataset split completed successfully!")
        print(f"  All {comparison['source']:,} samples from source were processed")
        print(f"  All splits are properly organized")
        print(f"  No data loss detected")
    else:
        print("\nWARNING: Verification issues detected!")
        print(f"  Please review the details above")

print("\n" + "-" * 70 + "\n")